In [1]:
import networkx as nx
import pandas as pd
import numpy as np
from rdflib import URIRef
import pickle

## Process NP-KG and reference data for embeddings pipeline

In [2]:
FILE_PATH = '/home/sanya/npkg-workspace/reference-data/mapped_processed_data/combined_interactions_nodrugclass_dedup_20240723.tsv'
DIR_OUT = 'data/reference-data/'

In [10]:
refdf = pd.read_csv(FILE_PATH, sep='\t')
refdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1455 entries, 0 to 1454
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id                1455 non-null   object
 1   np_id             1455 non-null   object
 2   np                1455 non-null   object
 3   drug_id           1455 non-null   object
 4   drug              1455 non-null   object
 5   interaction_type  1454 non-null   object
 6   source_data       1455 non-null   object
 7   adr               47 non-null     object
 8   adr_id            33 non-null     object
 9   protein_name      252 non-null    object
 10  related_np        1455 non-null   object
 11  related_np_id     1455 non-null   object
dtypes: object(12)
memory usage: 136.5+ KB


In [11]:
refdf.loc[refdf['interaction_type'].isna()]

,id,np_id,np,drug_id,drug,interaction_type,source_data,adr,adr_id,protein_name,related_np,related_np_id
1016,stockley_138,http://napdi.org/napdi_srs_imports:glycine_max,Soy,http://purl.obolibrary.org/obo/CHEBI_45863,Paclitaxel,NaN,Stockleys,NaN,NaN,NaN,Soy,http://napdi.org/napdi_srs_imports:glycine_max


In [12]:
refdf.at[1016, 'interaction_type'] = 'negative'

In [13]:
refdf['interaction_type'] = refdf['interaction_type'].str.lower()
refdf['interaction_type'].value_counts()

positive                                                                                                                                                                                                                                                                         785
positive|positive                                                                                                                                                                                                                                                                250
negative                                                                                                                                                                                                                                                                         119
positive|positive|positive|positive                                                                                                                                      

In [14]:
for i in range(len(refdf.index)):
    interaction_type_list = refdf.at[i, 'interaction_type'].split('|')
    interaction_type = list(set(interaction_type_list))
    if 'positive' in interaction_type and 'negative' in interaction_type:
        refdf.at[i, 'interaction_type'] = 'both'
    elif 'positive' in interaction_type:
        refdf.at[i, 'interaction_type'] = 'positive'
    elif 'negative' in interaction_type:
        refdf.at[i, 'interaction_type'] = 'negative'

In [15]:
refdf['interaction_type'].value_counts()

positive    1208
negative     161
both          86
Name: interaction_type, dtype: int64

In [16]:
##import graph
KG_PATH = '/home/sanya/np-kg/resources/knowledge_graphs/'
NodeLabelsFile = KG_PATH + 'nodeLabels_v3.0.0.pickle'
#KG_NAME_MERGED = 'NP-KG_v2.0.0.gpickle'
KG_NAME_MERGED_WEIGHTED = 'NP-KG_BETWEENNESS-WEIGHTED-PK-HASHED_v3.0.0.gpickle'
DIR_OUT = './output/'

In [17]:
with open(KG_PATH+KG_NAME_MERGED_WEIGHTED, 'rb') as f:
    npkg = pickle.load(f)

In [18]:
with open(NodeLabelsFile, 'rb') as filep:
    nodeLabels = pickle.load(filep)

In [19]:
def get_graph_stats(kg):
    nodes = nx.number_of_nodes(kg)
    edges = nx.number_of_edges(kg)
    self_loops = nx.number_of_selfloops(kg)

    print('There are {} nodes, {} edges, and {} self-loop(s)'.format(nodes, edges, self_loops))
    # get degree information
    avg_degree = float(edges) / nodes
    print('The Average Degree is {}'.format(avg_degree))
    
    print('Nodes with highest degree:')
    n_deg = sorted([(str(x[0]), x[1]) for x in  kg.degree], key=lambda x: x[1], reverse=1)[:6]

    for x in n_deg:
        print('Label: {}'.format(nodeLabels[x[0]]))
        print('{} (degree={})'.format(x[0], x[1]))
    # get network density
    density = nx.density(kg)

    print('The density of the graph is: {}'.format(density))

In [20]:
get_graph_stats(npkg)

There are 1089139 nodes, 7786937 edges, and 453 self-loop(s)
The Average Degree is 7.149626448047495
Nodes with highest degree:
Label: SNV
http://purl.obolibrary.org/obo/SO_0001483 (degree=214261)
Label: protein_coding_gene
http://purl.obolibrary.org/obo/SO_0001217 (degree=189350)
Label: transcript
http://purl.obolibrary.org/obo/SO_0000673 (degree=178309)
Label: Homo sapiens
http://purl.obolibrary.org/obo/NCBITaxon_9606 (degree=91837)
Label: protein
http://purl.obolibrary.org/obo/PR_000000001 (degree=56320)
Label: Mus musculus
http://purl.obolibrary.org/obo/NCBITaxon_10090 (degree=53597)
The density of the graph is: 6.5644816800511e-06


In [21]:
refdf.head()

,id,np_id,np,drug_id,drug,interaction_type,source_data,adr,adr_id,protein_name,related_np,related_np_id
0,napdi_88|napdi_89,http://purl.obolibrary.org/obo/CHEBI_16118,berberine|berberine,http://purl.obolibrary.org/obo/CHEBI_16469,17-beta-estradiol|17-beta-estradiol,positive,NaPDI|NaPDI,NaN,NaN,NaN,Hydrastis canadensis,http://napdi.org/napdi_srs_imports:hydrastis_c...
1,napdi_138,http://purl.obolibrary.org/obo/CHEBI_4806,epigallocatechin gallate,http://purl.obolibrary.org/obo/CHEBI_16469,17-beta-estradiol,positive,NaPDI,NaN,NaN,UGT1A1,Camellia sinensis,http://napdi.org/napdi_srs_imports:camellia_si...
2,napdi_9,http://purl.obolibrary.org/obo/CHEBI_69919,(-)-beta-hydrastine,http://purl.obolibrary.org/obo/CHEBI_791,17-beta-estradiol glucuronide,positive,NaPDI,NaN,NaN,OATP1B1 (SLCO1B1),Hydrastis canadensis,http://napdi.org/napdi_srs_imports:hydrastis_c...
3,napdi_219,http://napdi.org/napdi_srs_imports:hydrastis_c...,goldenseal root extract,http://purl.obolibrary.org/obo/CHEBI_791,17-beta-estradiol glucuronide,positive,NaPDI,NaN,NaN,OATP1B1 (SLCO1B1),Hydrastis canadensis,http://napdi.org/napdi_srs_imports:hydrastis_c...
4,napdi_15,http://purl.obolibrary.org/obo/CHEBI_90,(-)-epicatechin,http://purl.obolibrary.org/obo/CHEBI_17224,4-methylumbelliferone,negative,NaPDI,NaN,NaN,UGT,Camellia sinensis,http://napdi.org/napdi_srs_imports:camellia_si...


In [38]:
refdf['npkg_edge'] = pd.NA
refdf['npkg_edge_label'] = pd.NA

In [39]:
for i in range(len(refdf.index)):
    np_id = refdf.loc[i, 'np_id']
    drug_id = refdf.loc[i, 'drug_id']
    source = URIRef(np_id)
    target = URIRef(drug_id)
    if source in npkg and target in npkg:
        if npkg.has_edge(source, target):
            edge_data = npkg.get_edge_data(source, target)
            refdf.at[i, 'npkg_edge'] = edge_data['key']
refdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1455 entries, 0 to 1454
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id                1455 non-null   object
 1   np_id             1455 non-null   object
 2   np                1455 non-null   object
 3   drug_id           1455 non-null   object
 4   drug              1455 non-null   object
 5   interaction_type  1455 non-null   object
 6   source_data       1455 non-null   object
 7   adr               47 non-null     object
 8   adr_id            33 non-null     object
 9   protein_name      252 non-null    object
 10  related_np        1455 non-null   object
 11  related_np_id     1455 non-null   object
 12  npkg_edge         84 non-null     object
 13  npkg_edge_label   0 non-null      object
dtypes: object(14)
memory usage: 159.3+ KB


In [40]:
refdf.loc[refdf['npkg_edge'].notnull()]

,id,np_id,np,drug_id,drug,interaction_type,source_data,adr,adr_id,protein_name,related_np,related_np_id,npkg_edge,npkg_edge_label
4,napdi_15,http://purl.obolibrary.org/obo/CHEBI_90,(-)-epicatechin,http://purl.obolibrary.org/obo/CHEBI_17224,4-methylumbelliferone,negative,NaPDI,NaN,NaN,UGT,Camellia sinensis,http://napdi.org/napdi_srs_imports:camellia_si...,http://purl.obolibrary.org/obo/RO_0002213,<NA>
5,napdi_37,http://purl.obolibrary.org/obo/CHEBI_4806,(-)-epigallocatechin gallate,http://purl.obolibrary.org/obo/CHEBI_17224,4-methylumbelliferone,positive,NaPDI,NaN,NaN,UGT,Camellia sinensis,http://napdi.org/napdi_srs_imports:camellia_si...,http://purl.obolibrary.org/obo/RO_0002449,<NA>
8,napdi_127,http://purl.obolibrary.org/obo/CHEBI_70255,epicatechin gallate,http://purl.obolibrary.org/obo/CHEBI_17224,4-methylumbelliferone,positive,NaPDI,NaN,NaN,UGT,Camellia sinensis,http://napdi.org/napdi_srs_imports:camellia_si...,http://purl.obolibrary.org/obo/RO_0002449,<NA>
11,napdi_307|napdi_308|napdi_309,http://purl.obolibrary.org/obo/CHEBI_6956,mitragynine|mitragynine|mitragynine,http://purl.obolibrary.org/obo/CHEBI_17224,4-methylumbelliferone|4-methylumbelliferone|4-...,negative,NaPDI|NaPDI|NaPDI,NaN,NaN,NaN,Mitragyna speciosa,http://napdi.org/napdi_srs_imports:mitragyna_s...,http://purl.obolibrary.org/obo/RO_0002449,<NA>
34,ddid_226|ddid_233|ddid_240|ddid_247,http://purl.obolibrary.org/obo/CHEBI_27732,Caffeine |Caffeine |Caffeine |Caffeine,http://purl.obolibrary.org/obo/CHEBI_16335,Adenosine|Adenosine|Adenosine|Adenosine,positive,FDI-Package Insert|FDI-Package Insert|FDI-Pack...,NaN,NaN,NaN,Camellia sinensis [Syn. Thea sinensis],http://napdi.org/napdi_srs_imports:camellia_si...,http://purl.obolibrary.org/obo/RO_0002434|http...,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1318,napdi_318|napdi_320,http://purl.obolibrary.org/obo/CHEBI_6956,mitragynine|mitragynine,http://purl.obolibrary.org/obo/CHEBI_17347,testosterone|testosterone,negative,NaPDI|NaPDI,NaN,NaN,NaN,Mitragyna speciosa,http://napdi.org/napdi_srs_imports:mitragyna_s...,http://purl.obolibrary.org/obo/RO_0002449,<NA>
1324,natmed_158,http://napdi.org/napdi_srs_imports:camellia_si...,Green tea,http://purl.obolibrary.org/obo/CHEBI_28177,THEOPHYLLINE,positive,NatMedPro,NaN,NaN,NaN,Green tea,http://napdi.org/napdi_srs_imports:camellia_si...,http://purl.obolibrary.org/obo/RO_0002180,<NA>
1325,natmed_192,http://napdi.org/napdi_srs_imports:paullinia_c...,Guarana,http://purl.obolibrary.org/obo/CHEBI_28177,THEOPHYLLINE,positive,NatMedPro,NaN,NaN,NaN,Guarana,http://napdi.org/napdi_srs_imports:paullinia_c...,http://purl.obolibrary.org/obo/RO_0002180,<NA>
1423,fideo_310|natmed_16|stockley_174,http://napdi.org/napdi_srs_imports:camellia_si...,tea food product|Green tea|Green Tea,http://purl.obolibrary.org/obo/CHEBI_10033,warfarin|WARFARIN (Coumadin)|Warfarin and rela...,both,FIDEO|NatMedPro|Stockleys,bleeding,http://purl.obolibrary.org/obo/HP_0001892,NaN,tea food product,http://napdi.org/napdi_srs_imports:camellia_si...,http://purl.obolibrary.org/obo/RO_0002449,<NA>


In [42]:
for i in range(len(refdf.index)):
    npkg_edge = refdf.loc[i, 'npkg_edge']
    if not pd.isna(npkg_edge):
        edge_list = npkg_edge.split('|')
        if len(edge_list) == 1:
            refdf.at[i, 'npkg_edge_label'] = nodeLabels.get(edge_list[0], '')
        else:
            edge_label_list = [nodeLabels.get(x, '') for x in edge_list]
            refdf.at[i, 'npkg_edge_label'] = '|'.join(edge_label_list)
refdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1455 entries, 0 to 1454
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id                1455 non-null   object
 1   np_id             1455 non-null   object
 2   np                1455 non-null   object
 3   drug_id           1455 non-null   object
 4   drug              1455 non-null   object
 5   interaction_type  1455 non-null   object
 6   source_data       1455 non-null   object
 7   adr               47 non-null     object
 8   adr_id            33 non-null     object
 9   protein_name      252 non-null    object
 10  related_np        1455 non-null   object
 11  related_np_id     1455 non-null   object
 12  npkg_edge         84 non-null     object
 13  npkg_edge_label   84 non-null     object
dtypes: object(14)
memory usage: 159.3+ KB


In [48]:
refdf.to_csv(DIR_OUT+'reference_data_combined_interactions_nodrugclass_dedup_20240724_with_npkg_edges.tsv', sep='\t', index=False)

In [49]:
refdf_nokg = refdf.loc[refdf['npkg_edge'].isna()]
refdf_nokg = refdf_nokg.drop(columns=['npkg_edge', 'npkg_edge_label'])
refdf_nokg = refdf_nokg.reset_index(drop=True)
refdf_nokg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1371 entries, 0 to 1370
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id                1371 non-null   object
 1   np_id             1371 non-null   object
 2   np                1371 non-null   object
 3   drug_id           1371 non-null   object
 4   drug              1371 non-null   object
 5   interaction_type  1371 non-null   object
 6   source_data       1371 non-null   object
 7   adr               45 non-null     object
 8   adr_id            32 non-null     object
 9   protein_name      238 non-null    object
 10  related_np        1371 non-null   object
 11  related_np_id     1371 non-null   object
dtypes: object(12)
memory usage: 128.7+ KB


In [50]:
refdf_nokg.to_csv(DIR_OUT+'reference_data_combined_interactions_nodrugclass_dedup_20240724_without_npkg_edges.tsv', sep='\t', index=False)

### Load data and convert to CURIEs -- save only edges for prediction

In [3]:
refdf = pd.read_csv(DIR_OUT+'reference_data_combined_interactions_nodrugclass_dedup_20240724_without_npkg_edges.tsv', sep='\t')

In [4]:
refdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1371 entries, 0 to 1370
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id                1371 non-null   object
 1   np_id             1371 non-null   object
 2   np                1371 non-null   object
 3   drug_id           1371 non-null   object
 4   drug              1371 non-null   object
 5   interaction_type  1371 non-null   object
 6   source_data       1371 non-null   object
 7   adr               45 non-null     object
 8   adr_id            32 non-null     object
 9   protein_name      238 non-null    object
 10  related_np        1371 non-null   object
 11  related_np_id     1371 non-null   object
dtypes: object(12)
memory usage: 128.7+ KB


In [7]:
refdf.head()

,id,np_id,np,drug_id,drug,interaction_type,source_data,adr,adr_id,protein_name,related_np,related_np_id
0,napdi_88|napdi_89,http://purl.obolibrary.org/obo/CHEBI_16118,berberine|berberine,http://purl.obolibrary.org/obo/CHEBI_16469,17-beta-estradiol|17-beta-estradiol,positive,NaPDI|NaPDI,NaN,NaN,NaN,Hydrastis canadensis,http://napdi.org/napdi_srs_imports:hydrastis_c...
1,napdi_138,http://purl.obolibrary.org/obo/CHEBI_4806,epigallocatechin gallate,http://purl.obolibrary.org/obo/CHEBI_16469,17-beta-estradiol,positive,NaPDI,NaN,NaN,UGT1A1,Camellia sinensis,http://napdi.org/napdi_srs_imports:camellia_si...
2,napdi_9,http://purl.obolibrary.org/obo/CHEBI_69919,(-)-beta-hydrastine,http://purl.obolibrary.org/obo/CHEBI_791,17-beta-estradiol glucuronide,positive,NaPDI,NaN,NaN,OATP1B1 (SLCO1B1),Hydrastis canadensis,http://napdi.org/napdi_srs_imports:hydrastis_c...
3,napdi_219,http://napdi.org/napdi_srs_imports:hydrastis_c...,goldenseal root extract,http://purl.obolibrary.org/obo/CHEBI_791,17-beta-estradiol glucuronide,positive,NaPDI,NaN,NaN,OATP1B1 (SLCO1B1),Hydrastis canadensis,http://napdi.org/napdi_srs_imports:hydrastis_c...
4,napdi_41,http://purl.obolibrary.org/obo/CHEBI_31018,(-)-gallocatechin,http://purl.obolibrary.org/obo/CHEBI_17224,4-methylumbelliferone,negative,NaPDI,NaN,NaN,UGT,Camellia sinensis,http://napdi.org/napdi_srs_imports:camellia_si...


In [24]:
uri_to_curie_map_file = '/home/sanya/np-kg/resources/knowledge_graphs/uri_to_curie_map.tsv'
##load as dictionary
uri_to_curie_map = {}
with open(uri_to_curie_map_file, 'r') as f:
    for line in f:
        uri, curie = line.strip().split('\t')
        uri_to_curie_map[uri] = curie
len(uri_to_curie_map)

1089472

In [25]:
for key in uri_to_curie_map:
    print(key, uri_to_curie_map[key])
    break

http://purl.obolibrary.org/obo/CLO_0035452 clo:0035452


In [26]:
##add missing curies
uri_to_curie_map['http://purl.obolibrary.org/obo/CHEBI:9654'] = 'chebi:9654'

In [27]:
edge_df = pd.DataFrame(columns=['source', 'target', 'interaction_type'])
for i in range(len(refdf.index)):
    np_id = refdf.at[i, 'np_id']
    drug_id = refdf.at[i, 'drug_id']
    interaction_type = refdf.at[i, 'interaction_type']
    if np_id in uri_to_curie_map and drug_id in uri_to_curie_map:
        source = uri_to_curie_map[np_id]
        target = uri_to_curie_map[drug_id]
        tempdf = pd.DataFrame({'source': [source], 'target': [target], 'interaction_type': [interaction_type]})
        edge_df = pd.concat([edge_df, tempdf], ignore_index=True)
    else:
        print('URI not found in map: {} {}'.format(np_id, drug_id))
edge_df.info()

URI not found in map: http://napdi.org/napdi_srs_imports:serenoa_repens http://purl.obolibrary.org/obo/CHEBI_4475
URI not found in map: http://napdi.org/napdi_srs_imports:valeriana_officinalis http://purl.obolibrary.org/obo/CHEBI_4476
URI not found in map: http://napdi.org/napdi_srs_imports:citrus_paradisi http://purl.obolibrary.org/obo/CHEBI_229213
URI not found in map: http://purl.obolibrary.org/obo/CHEBI_207762 http://purl.obolibrary.org/obo/CHEBI_8772
URI not found in map: http://napdi.org/napdi_srs_imports:citrus_paradisi http://purl.obolibrary.org/obo/CHEBI_230905
URI not found in map: http://napdi.org/napdi_srs_imports:citrus_paradisi http://purl.obolibrary.org/obo/CHEBI_94311
URI not found in map: http://napdi.org/napdi_srs_imports:citrus_paradisi http://purl.obolibrary.org/obo/CHEBI_229526
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1364 entries, 0 to 1363
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            -------------- 

In [29]:
edge_df.head()

,source,target,interaction_type
0,chebi:16118,chebi:16469,positive
1,chebi:4806,chebi:16469,positive
2,chebi:69919,chebi:791,positive
3,napdi_srs_imports:hydrastis_canadensis,chebi:791,positive
4,chebi:31018,chebi:17224,negative


In [30]:
edge_df.to_csv(DIR_OUT+'reference_data_curie_edges_20240725.tsv', sep='\t', index=False)

In [21]:
####check if new nodes exist in KG
import pickle
import networkx as nx
from rdflib import URIRef

In [19]:
KG_PATH = '/home/sanya/np-kg/resources/knowledge_graphs/'
KG_NAME = 'NP-KG_BETWEENNESS-WEIGHTED-PK-HASHED_v3.0.0.gpickle'

In [20]:
with open(KG_PATH+KG_NAME, 'rb') as f:
    npkg = pickle.load(f)

In [23]:
nodelist = ['http://purl.obolibrary.org/obo/CHEBI_4475','http://purl.obolibrary.org/obo/CHEBI_4476','http://purl.obolibrary.org/obo/CHEBI_229213',
'http://purl.obolibrary.org/obo/CHEBI_207762','http://purl.obolibrary.org/obo/CHEBI_230905','http://purl.obolibrary.org/obo/CHEBI_94311',
'http://purl.obolibrary.org/obo/CHEBI_229526','http://purl.obolibrary.org/obo/CHEBI_9654']

for node in nodelist:
    if URIRef(node) not in npkg:
        print(node)

http://purl.obolibrary.org/obo/CHEBI_4475
http://purl.obolibrary.org/obo/CHEBI_4476
http://purl.obolibrary.org/obo/CHEBI_229213
http://purl.obolibrary.org/obo/CHEBI_207762
http://purl.obolibrary.org/obo/CHEBI_230905
http://purl.obolibrary.org/obo/CHEBI_94311
http://purl.obolibrary.org/obo/CHEBI_229526
